# 02 — A/B Test Analysis

**Pre-registered decision:** D30 retention is the primary endpoint; D1 and D7 are guardrails/diagnostics. Use a two-sided 5% test, 80% power and 1.14 percentage-point absolute MDE. Read out only after the final assigned cohort has 30 complete days. No efficacy peeking before that fixed date; run SRM, data-quality and safety checks continuously. Apply Holm correction across D1/D7/D30.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analytics import srm_check, experiment_design, retention_readout, segment_retention, business_scenarios
DATA = ROOT / 'data'
players = pd.read_csv(DATA / 'players.csv', parse_dates=['install_date'])
sessions = pd.read_csv(DATA / 'sessions.csv', parse_dates=['session_start'])
transactions = pd.read_csv(DATA / 'iap_transactions.csv', parse_dates=['transaction_timestamp'])
players = players.loc[players.acquisition_channel.ne('internal_qa')].copy()
sessions = sessions.loc[sessions.player_id.isin(players.player_id)]
print('SRM:', srm_check(players))
print('Sample-size design:', experiment_design())
retention, readout = retention_readout(players, sessions)
display(readout)

## Commercial decision scenarios

All options are projected to the same eligible population size. The D30 revenue-at-risk figure uses control 30-day ARPU as a proxy per lost retained player; the observed ARPU gap is reported separately and is not attributed solely to retention.

In [ ]:
display(business_scenarios(players, sessions, transactions))

## Reversal segmentation

Inspect country, platform, acquisition channel and spend tier. Treat small segments as exploratory; apply multiplicity control before making segment-level rollout decisions.

In [ ]:
segments = segment_retention(players, sessions)
display(segments.sort_values(['dimension','day','segment','variant']))